In [3]:
import os
# 防止 Tokenizer 多进程死锁
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import pandas as pd
from eval_jsonl import eval_jsonl_fast
from tqdm.auto import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

# 1. 按照新规则获取并过滤文件路径
file_paths = ["/workspace/yiqiuguo/lsrl/gen_results/"+x for x in os.listdir("/workspace/yiqiuguo/lsrl/gen_results/")]
file_paths = [
    p for p in file_paths if \
    ("qwen2.5-1.5b" in p or (p.split('/')[-1].startswith('step') and "step688" not in p)) and \
    ("math-500" in p or "aime" in p or "amc23" in p) and \
    ("rollout32" in p) and \
    ("step109" in p or "step108" in p or "step114" in p or "qwen2.5-1.5b" in p)
]

# 2. 定义单任务包装函数
def process_file(path):
    try:
        res_dict = eval_jsonl_fast(path,possible_ks=[1,8,32])
        return path, res_dict, None
    except Exception as e:
        return path, None, str(e)

# 3. 启用 64 进程进行加速
results = []
print(f"Starting ProcessPoolExecutor with 64 workers for {len(file_paths)} files...")

with ProcessPoolExecutor(max_workers=64) as executor:
    futures = [executor.submit(process_file, path) for path in file_paths]
    for future in tqdm(as_completed(futures), total=len(futures), desc="Evaluating JSONL"):
        path, res_dict, error = future.result()
        if error:
            print(f"Error processing {path}: {error}")
        elif res_dict is not None:
            results.append(res_dict)

# 4. 数据透视与整理
df = pd.DataFrame(results)

# 提取并排序所有的 Pass@k 列
pass_cols = sorted([c for c in df.columns if 'Pass@' in c], key=lambda x: int(x.split('@')[1].split(' ')[0]))
df_multi = df.pivot_table(index='Model', columns='Dataset', values=pass_cols, aggfunc='mean')
df_multi = df_multi.swaplevel(axis=1)

# 重组双层表头
datasets = sorted(df['Dataset'].dropna().unique())
ordered_columns = pd.MultiIndex.from_product([datasets, pass_cols], names=['Dataset', 'Pass@k'])
df_multi = df_multi.reindex(columns=ordered_columns)

# 5. 【核心逻辑】计算绝对与相对提升，生成带有格式的字符串数据框
df_display = df_multi.copy().astype(object)

# 寻找基线模型 (包含 qwen2.5-1.5b 的那一行)
baseline_idx = next((idx for idx in df_multi.index if 'qwen2.5-1.5b' in str(idx)), None)

for col in df_multi.columns:
    base_val = df_multi.loc[baseline_idx, col] if (baseline_idx and baseline_idx in df_multi.index) else None
    
    for idx in df_multi.index:
        val = df_multi.loc[idx, col]
        
        if pd.isna(val):
            df_display.loc[idx, col] = "-"
            continue
            
        # 如果是基线模型，标记为 Base
        if idx == baseline_idx:
            df_display.loc[idx, col] = f"{val:.2f}"
        # 计算提升值
        elif base_val is not None and not pd.isna(base_val):
            abs_diff = val - base_val
            rel_diff = (abs_diff / base_val) * 100 if base_val != 0 else 0
            
            # 添加正负号
            abs_sign = "+" if abs_diff > 0 else ""
            rel_sign = "+" if rel_diff > 0 else ""
            
            # 格式例如: 82.50 (+2.50, +3.12%)
            df_display.loc[idx, col] = f"{val:.2f} ({abs_sign}{abs_diff:.2f}, {rel_sign}{rel_diff:.2f}%)"
        else:
            df_display.loc[idx, col] = f"{val:.2f}"

# 6. 【核心样式】自定义函数寻找第一名与第二名
def highlight_top2(data, df_numeric):
    # data 是字符串的 df_display，但我们需要用 df_numeric 来比较大小
    css_df = pd.DataFrame('', index=data.index, columns=data.columns)
    
    for col in df_numeric.columns:
        series = df_numeric[col].dropna()
        if series.empty:
            continue
        
        # 获取所有唯一值并排序 (从小到大)
        unique_vals = sorted(series.unique())
        
        # 取最大的和第二大的数值
        max_val = unique_vals[-1] if len(unique_vals) >= 1 else None
        second_max = unique_vals[-2] if len(unique_vals) >= 2 else None
        
        for idx in series.index:
            val = series[idx]
            css_classes = []
            
            if val == max_val and max_val is not None:
                css_classes.append('font-weight: bold')
            if val == second_max and second_max is not None:
                css_classes.append('text-decoration: underline')
            
            if css_classes:
                css_df.loc[idx, col] = '; '.join(css_classes)
                
    return css_df

# 应用样式并渲染展示
styled_df = df_display.style\
    .apply(lambda x: highlight_top2(x, df_multi), axis=None)\
    .set_properties(**{'text-align': 'center'})

display(styled_df)

Starting ProcessPoolExecutor with 64 workers for 16 files...


Evaluating JSONL:   0%|          | 0/16 [00:00<?, ?it/s]

In [1]:
import os
# 防止 Tokenizer 多进程死锁
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import pandas as pd
from eval_jsonl import eval_jsonl_fast
from tqdm.auto import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

# 1. 按照新规则获取并过滤文件路径 (只读取指定模型)
target_models = ["qwen2.5-1.5b", "step108", "step109", "step114"]

file_paths = ["/workspace/yiqiuguo/lsrl/gen_results/"+x for x in os.listdir("/workspace/yiqiuguo/lsrl/gen_results/")]
file_paths = [
    p for p in file_paths if \
    any(m in p for m in target_models) and \
    ("math-500" in p or "aime" in p or "amc23" in p) and \
    ("rollout32" in p) and \
    ("node" not in p) and \
    ("0430" in p)
]
for d in file_paths:
    print(d)
# 2. 定义单任务包装函数
def process_file(path):
    try:
        res_dict = eval_jsonl_fast(path, possible_ks=[1,8,32], return_length=False)
        return path, res_dict, None
    except Exception as e:
        return path, None, str(e)

# 3. 启用 64 进程进行加速
results = []
print(f"Starting ProcessPoolExecutor with 64 workers for {len(file_paths)} files...")

with ProcessPoolExecutor(max_workers=64) as executor:
    futures = [executor.submit(process_file, path) for path in file_paths]
    for future in tqdm(as_completed(futures), total=len(futures), desc="Evaluating JSONL"):
        path, res_dict, error = future.result()
        if error:
            print(f"Error processing {path}: {error}")
        elif res_dict is not None:
            results.append(res_dict)

# 4. 数据透视与整理
df = pd.DataFrame(results)

# 替换模型名称
def rename_model(m):
    m_str = str(m)
    if "step109" in m_str: return "all_hard"
    if "step114" in m_str: return "conn_hard"
    if "step108" in m_str: return "opsd"
    if "qwen2.5-1.5b" in m_str: return "qwen2.5-1.5b"
    return m_str

df['Model'] = df['Model'].apply(rename_model)

# 提取并排序所有的 Pass@k 列
pass_cols = sorted([c for c in df.columns if 'Pass@' in c], key=lambda x: int(x.split('@')[1].split(' ')[0]))
df_multi = df.pivot_table(index='Model', columns='Dataset', values=pass_cols, aggfunc='mean')
df_multi = df_multi.swaplevel(axis=1)

# 重组双层表头
datasets = sorted(df['Dataset'].dropna().unique())
ordered_columns = pd.MultiIndex.from_product([datasets, pass_cols], names=['Dataset', 'Pass@k'])
df_multi = df_multi.reindex(columns=ordered_columns)

# 5. 【核心逻辑】计算绝对与相对提升，并统计平均提升值
df_display = df_multi.copy().astype(object)

# 寻找基线模型
baseline_idx = next((idx for idx in df_multi.index if 'qwen2.5-1.5b' in str(idx)), None)

# 用于统计平均提升值的字典
avg_abs_sum = {idx: 0.0 for idx in df_multi.index}
avg_rel_sum = {idx: 0.0 for idx in df_multi.index}
valid_count = {idx: 0 for idx in df_multi.index}

for col in df_multi.columns:
    base_val = df_multi.loc[baseline_idx, col] if (baseline_idx and baseline_idx in df_multi.index) else None
    
    for idx in df_multi.index:
        val = df_multi.loc[idx, col]
        
        if pd.isna(val):
            df_display.loc[idx, col] = "-"
            continue
            
        # 如果是基线模型，标记为 Base
        if idx == baseline_idx:
            df_display.loc[idx, col] = f"{val:.2f}"
        # 计算提升值
        elif base_val is not None and not pd.isna(base_val):
            abs_diff = val - base_val
            rel_diff = (abs_diff / base_val) * 100 if base_val != 0 else 0
            
            # 添加正负号
            abs_sign = "+" if abs_diff > 0 else ""
            rel_sign = "+" if rel_diff > 0 else ""
            
            # 格式例如: 82.50 (+2.50, +3.12%)
            df_display.loc[idx, col] = f"{val:.2f} ({abs_sign}{abs_diff:.2f}, {rel_sign}{rel_diff:.2f}%)"
            
            # 累加用于计算平均值
            avg_abs_sum[idx] += abs_diff
            avg_rel_sum[idx] += rel_diff
            valid_count[idx] += 1
        else:
            df_display.loc[idx, col] = f"{val:.2f}"

# 插入平均提升列
for idx in df_multi.index:
    if idx == baseline_idx:
        df_display.loc[idx, ('Average', 'Improvement')] = "Base"
    else:
        cnt = valid_count[idx]
        if cnt > 0:
            mean_abs = avg_abs_sum[idx] / cnt
            mean_rel = avg_rel_sum[idx] / cnt
            m_abs_sign = "+" if mean_abs > 0 else ""
            m_rel_sign = "+" if mean_rel > 0 else ""
            df_display.loc[idx, ('Average', 'Improvement')] = f"{m_abs_sign}{mean_abs:.2f}, {m_rel_sign}{mean_rel:.2f}%"
        else:
            df_display.loc[idx, ('Average', 'Improvement')] = "-"

# 6. 【核心样式】自定义函数寻找第一名与第二名
def highlight_top2(data, df_numeric):
    css_df = pd.DataFrame('', index=data.index, columns=data.columns)
    
    for col in df_numeric.columns:
        # 只针对存在于 df_numeric 的列进行最高分计算（跳过 Average Improvement 这种纯字符串列）
        if col not in df_numeric:
            continue
            
        series = df_numeric[col].dropna()
        if series.empty:
            continue
        
        unique_vals = sorted(series.unique())
        
        max_val = unique_vals[-1] if len(unique_vals) >= 1 else None
        second_max = unique_vals[-2] if len(unique_vals) >= 2 else None
        
        for idx in series.index:
            val = series[idx]
            css_classes = []
            
            if val == max_val and max_val is not None:
                css_classes.append('font-weight: bold')
            if val == second_max and second_max is not None:
                css_classes.append('text-decoration: underline')
            
            if css_classes:
                css_df.loc[idx, col] = '; '.join(css_classes)
                
    return css_df

# 应用样式并渲染展示
styled_df = df_display.style\
    .apply(lambda x: highlight_top2(x, df_multi), axis=None)\
    .set_properties(**{'text-align': 'center'})

display(styled_df)

/workspace/yiqiuguo/lsrl/gen_results/qwen2.5-1.5b-instruct_aime25_rollout32_run20260430_3871.jsonl
/workspace/yiqiuguo/lsrl/gen_results/qwen2.5-1.5b-instruct_aime_2024_rollout32_run20260430_3871.jsonl
/workspace/yiqiuguo/lsrl/gen_results/qwen2.5-1.5b-instruct_amc23_rollout32_run20260430_3871.jsonl
/workspace/yiqiuguo/lsrl/gen_results/qwen2.5-1.5b-instruct_math-500_rollout32_run20260430_3871.jsonl
/workspace/yiqiuguo/lsrl/gen_results/step109_aime25_rollout32_run20260430_3871.jsonl
/workspace/yiqiuguo/lsrl/gen_results/step109_aime_2024_rollout32_run20260430_3871.jsonl
/workspace/yiqiuguo/lsrl/gen_results/step109_amc23_rollout32_run20260430_3871.jsonl
/workspace/yiqiuguo/lsrl/gen_results/step109_math-500_rollout32_run20260430_3871.jsonl
Starting ProcessPoolExecutor with 64 workers for 8 files...


Evaluating JSONL:   0%|          | 0/8 [00:00<?, ?it/s]